# Validación de `AccumulatedAnnualDemand` (TRA*): Nacional vs. Regional

Este notebook compara el parámetro `AccumulatedAnnualDemand` para los combustibles de transporte (prefijo `TRA`) entre:

- **Nacional**: `SAND_Nacional_base/01-04-2026 SAND BASE v10.xlsx`
- **Regional**: `SAND_Regional/scenario_23_Parameters_SAND.xlsx` (7 regiones: CA, OR, SO, AN, NE, SE, IN, con formato `PREFIJO_FUEL`)

El objetivo es sumar las 7 regiones por combustible y año, y compararlas contra el valor nacional para detectar discrepancias.

In [ ]:
import re
import pandas as pd

pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', lambda v: f'{v:,.6f}')

In [ ]:
# --- Configuración ---
NACIONAL_PATH = "SAND_Nacional_base/01-04-2026 SAND BASE v10.xlsx"
REGIONAL_PATH = "SAND_Regional/scenario_23_Parameters_SAND.xlsx"

PARAMETER = "AccumulatedAnnualDemand"
FUEL_ROOT = "TRA"
REGIONES_VALIDAS = {"CA", "OR", "SO", "AN", "NE", "SE", "IN"}
TOLERANCIA = 1e-4

In [ ]:
# --- Carga de archivos ---
df_nacional_raw = pd.read_excel(NACIONAL_PATH, sheet_name="Parameters")
df_regional_raw = pd.read_excel(REGIONAL_PATH, sheet_name="Parameters")

print("Nacional:", df_nacional_raw.shape)
print("Regional:", df_regional_raw.shape)

In [ ]:
# --- Columnas de año (formato ancho) ---
YEAR_COLS = [c for c in df_nacional_raw.columns if re.fullmatch(r"\d{4}", str(c))]
print(f"{len(YEAR_COLS)} columnas de año detectadas: {YEAR_COLS[0]}..{YEAR_COLS[-1]}")

ID_COLS = ["Parameter", "FUEL"]

## 1. Filtrado y transformación a formato largo — Nacional

In [ ]:
df_nac_filt = df_nacional_raw[
    (df_nacional_raw["Parameter"] == PARAMETER)
    & (df_nacional_raw["FUEL"].astype(str).str.startswith(FUEL_ROOT))
].copy()

df_nac_long = df_nac_filt[ID_COLS + YEAR_COLS].melt(
    id_vars=ID_COLS, value_vars=YEAR_COLS, var_name="YEAR", value_name="VALUE"
)
df_nac_long["YEAR"] = df_nac_long["YEAR"].astype(int)
df_nac_long["VALUE"] = pd.to_numeric(df_nac_long["VALUE"], errors="coerce").fillna(0)

print("Combustibles TRA* encontrados en Nacional:", sorted(df_nac_long['FUEL'].unique()))
df_nac_long.head()

## 2. Filtrado, limpieza de prefijos y agregación — Regional

In [ ]:
df_reg_filt = df_regional_raw[df_regional_raw["Parameter"] == PARAMETER].copy()

# Separar PREFIJO_FUEL -> (REGION_PREFIX, FUEL)
fuel_split = df_reg_filt["FUEL"].astype(str).str.split("_", n=1, expand=True)
df_reg_filt["REGION_PREFIX"] = fuel_split[0]
df_reg_filt["FUEL"] = fuel_split[1]

# Validar que todos los prefijos correspondan a las 7 regiones esperadas
prefijos_inesperados = set(df_reg_filt["REGION_PREFIX"].dropna().unique()) - REGIONES_VALIDAS
if prefijos_inesperados:
    print(f"ADVERTENCIA: prefijos de región inesperados encontrados: {prefijos_inesperados}")

# Filtrar por raiz TRA sobre el FUEL ya limpio
df_reg_filt = df_reg_filt[df_reg_filt["FUEL"].astype(str).str.startswith(FUEL_ROOT)]

df_reg_long = df_reg_filt[ID_COLS + YEAR_COLS].melt(
    id_vars=ID_COLS, value_vars=YEAR_COLS, var_name="YEAR", value_name="VALUE"
)
df_reg_long["YEAR"] = df_reg_long["YEAR"].astype(int)
df_reg_long["VALUE"] = pd.to_numeric(df_reg_long["VALUE"], errors="coerce").fillna(0)

print("Combustibles TRA* encontrados en Regional (sin prefijo):", sorted(df_reg_long['FUEL'].unique()))
df_reg_long.head()

In [ ]:
# Consolidar las 7 regiones: suma por FUEL y YEAR
df_reg_grouped = (
    df_reg_long.groupby(["FUEL", "YEAR"], as_index=False)["VALUE"]
    .sum()
    .rename(columns={"VALUE": "VALUE_REGIONAL"})
)
df_reg_grouped.head()

## 3. Cruce (merge) y cálculo de diferencias

In [ ]:
df_nac_agg = df_nac_long.rename(columns={"VALUE": "VALUE_NACIONAL"})[["FUEL", "YEAR", "VALUE_NACIONAL"]]

df_comparacion = pd.merge(
    df_nac_agg, df_reg_grouped, on=["FUEL", "YEAR"], how="outer"
)

# Manejo de nulos: combinaciones FUEL-YEAR presentes en un archivo pero no en el otro
df_comparacion["VALUE_NACIONAL"] = df_comparacion["VALUE_NACIONAL"].fillna(0)
df_comparacion["VALUE_REGIONAL"] = df_comparacion["VALUE_REGIONAL"].fillna(0)

df_comparacion["DELTA"] = df_comparacion["VALUE_NACIONAL"] - df_comparacion["VALUE_REGIONAL"]
df_comparacion["DELTA_ABS"] = df_comparacion["DELTA"].abs()

df_comparacion = df_comparacion.sort_values(["FUEL", "YEAR"]).reset_index(drop=True)
df_comparacion.head(20)

## 4. Validación de tolerancia y alertas

In [ ]:
discrepancias = df_comparacion[df_comparacion["DELTA_ABS"] > TOLERANCIA].copy()
discrepancias = discrepancias.sort_values("DELTA_ABS", ascending=False)

if not discrepancias.empty:
    print(
        f"ALERTA: se encontraron {len(discrepancias)} combinaciones FUEL-YEAR "
        f"con diferencia absoluta mayor a la tolerancia ({TOLERANCIA}).\n"
    )
    print(f"Combustibles afectados: {sorted(discrepancias['FUEL'].unique())}")
    print(f"Diferencia maxima observada: {discrepancias['DELTA_ABS'].max():,.6f}")
else:
    print(f"OK: todas las diferencias estan dentro de la tolerancia ({TOLERANCIA}).")

discrepancias

## 5. Resumen por combustible

In [ ]:
resumen_por_fuel = (
    df_comparacion.groupby("FUEL")
    .agg(
        delta_abs_max=("DELTA_ABS", "max"),
        delta_abs_promedio=("DELTA_ABS", "mean"),
        anios_con_discrepancia=("DELTA_ABS", lambda s: (s > TOLERANCIA).sum()),
    )
    .sort_values("delta_abs_max", ascending=False)
)
resumen_por_fuel